# Observability with OpenTelemetry

Pixeltable can export its internal telemetry as OpenTelemetry (OTel) spans, so you can watch
operations in any OTel backend. This notebook wires the OTel bridge to **Grafana Cloud** and traces a
single `insert()` that exercises the full span set: source preparation, lock acquisition, planning,
external media downloads, UDF evaluation, media persistence, the store write path, and view propagation.

The insert emits one nested span tree:

```
pixeltable.insert                    (operation span, root)
├─ pixeltable.data_source.prepare    (source resolution + schema validation)
├─ pixeltable.catalog.begin_xact     (connection + lock acquisition, one per attempt)
├─ pixeltable.plan.create            (insert plan construction, DEBUG level)
├─ pixeltable.media.fetch            (one per external file downloaded, DEBUG level)
├─ pixeltable.row                    (one per inserted row, DEBUG level)
│  └─ pixeltable.udf.<name>          (each UDF call, nested under its row)
├─ pixeltable.media.save             (one per generated media file persisted, DEBUG level)
├─ pixeltable.store.build_rows       (row -> store-row conversion, DEBUG level)
├─ pixeltable.sa.insert_rows         (the SQL INSERT)
└─ pixeltable.view_load              (one per mutable view, containing its own row/store spans)
```

Traces land in Grafana Cloud Traces (Tempo).

In [ ]:
%pip install -qU 'pixeltable[otel]'

## Turn on the OpenTelemetry bridge

Telemetry is opt-in: you must call `pxt_otel.init()` once. It reads the standard `OTEL_EXPORTER_OTLP_*`
env vars if present, otherwise we build the endpoint and Basic-auth header from the `GRAFANA_*` vars.

`init()` runs **once per process**. If it ran under a bad config earlier in this kernel it can't be
reconfigured, so restart the kernel and run this cell first.

In [1]:
import base64
import opentelemetry.instrumentation.pixeltable as pxt_otel
import os
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from pixeltable import exceptions as excs, hooks

# init() is ONCE PER PROCESS. If it already ran in this kernel under the wrong config, re-running is a
# no-op (it raises below) -- restart the kernel (Kernel -> Restart) to reconfigure.
try:
    if os.environ.get('OTEL_EXPORTER_OTLP_ENDPOINT'):
        pxt_otel.init()  # resolve endpoint/headers natively from the OTEL_EXPORTER_OTLP_* env vars
    else:
        # configure explicitly from the GRAFANA_* vars -- does not rely on the kernel inheriting OTEL_* vars
        auth = base64.b64encode(
            f'{os.environ["GRAFANA_OTLP_INSTANCE_ID"]}:{os.environ["GRAFANA_SERVICE_ACCOUNT_TOKEN"]}'.encode()
        ).decode()
        pxt_otel.init(
            endpoint=os.environ['GRAFANA_OTLP_URL'],
            headers=f'Authorization=Basic {auth}',
        )
except excs.Error as e:
    print(f'init() skipped: {e}')

hooks.set_span_level(
    hooks.DEBUG
)  # emit per-row + per-UDF spans (default INFO = operation span only)

# fail loud instead of silently dropping every span: a no-op ProxyTracerProvider means the SDK never came up
assert isinstance(trace.get_tracer_provider(), TracerProvider), (
    'OpenTelemetry is INERT -- no OTLP exporter was configured on the first init() in this kernel. '
    'Restart the kernel and run this cell FIRST, before any table operation.'
)
print('OTel bridge active -> Grafana Cloud')

OTel bridge active -> Grafana Cloud


## A pipeline that exercises every span type

One table with an image column and two computed columns, plus a view:

- inserting external image URLs triggers `pixeltable.media.fetch` (one download span per file, on worker threads)
- the `add_one` UDF produces a `pixeltable.udf.add_one` span under each `pixeltable.row`
- the stored `thumb` column generates thumbnails whose persistence produces `pixeltable.media.save` spans
- the view is recomputed as part of the same insert, producing the nested `pixeltable.view_load` subtree

`if_exists='replace_force'` also drops the dependent view on re-runs.

In [2]:
import pixeltable as pxt


@pxt.udf
def add_one(x: int) -> int:
    return x + 1


t = pxt.create_table(
    'otel_demo1',
    {'c': pxt.Int, 'img': pxt.Image},
    if_exists='replace_force',
)
t.add_computed_column(inc=add_one(t.c))
t.add_computed_column(thumb=t.img.resize((64, 64)))
v = pxt.create_view(
    'otel_demo_view',
    t,
    additional_columns={'inc2': add_one(t.inc)},
    if_exists='replace',
)

Created table 'otel_demo1'.
Added 0 column values with 0 errors in 0.01 s
Added 0 column values with 0 errors in 0.00 s


## Insert

This is the traced operation: 4 rows referencing external images (well under the 100-span-per-operation
cap, so every row gets a span). The whole pipeline, downloads, UDFs, thumbnails, store writes, and the
view reload, lands under one `pixeltable.insert` trace.

Note: `pixeltable.media.fetch` spans only appear when the files are not yet in the local file cache; on a re-run of
this cell the downloads are skipped.

In [3]:
base = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images'
images = [
    '000000000001.jpg',
    '000000000016.jpg',
    '000000000019.jpg',
    '000000000025.jpg',
]
status = t.insert(
    [{'c': i, 'img': f'{base}/{name}'} for i, name in enumerate(images)]
)
status

Inserted 8 rows with 0 errors in 0.57 s (13.97 rows/s)


8 rows inserted.

## Wait a few seconds for export

`init()` sets up a `BatchSpanProcessor`, which ships queued spans automatically about every 5 seconds while the kernel stays alive, so no manual flush is needed. Give it a few seconds after the insert, then check Grafana.

(In a short-lived script that exits immediately you would call `trace.get_tracer_provider().force_flush()` before exit, but in a live notebook kernel the timer handles it.)

## View it in Grafana

- **Traces**: Explore -> your Traces (Tempo) data source -> search Service Name `pixeltable`, or span name
  `pixeltable.insert`. Expand the trace to see the full hierarchy: `pixeltable.media.fetch` downloads running in
  parallel, `pixeltable.row` -> `pixeltable.udf.add_one`/`pixeltable.udf.resize` nesting, `pixeltable.media.save`
  persistence, the `pixeltable.store.build_rows`/`pixeltable.sa.insert_rows` write path, and the
  `pixeltable.view_load` subtree with its own rows and store writes.

For more detail raise the span level (`hooks.set_span_level(hooks.TRACE)`); to reduce volume on a large
insert, leave it at the default INFO (operation, `data_source.prepare`, `begin_xact`, `sa.insert_rows`,
and `view_load` spans only).